In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 1. Load data
df = pd.read_csv('marketing_AB.csv')

In [3]:
df.head(5)

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB


In [5]:
# Drop unnecessary index columns
df = df.drop(columns=['Unnamed: 0'])

In [6]:
# In Kaggle dataset: 'test' vs 'psa' (or 'control'), and 'converted' (True/False)
group_summary = df.groupby('test group').agg(
    total_users=('user id', 'count'),
    converted_users=('converted', 'sum')
).reset_index()

group_summary['conversion_rate'] = group_summary['converted_users'] / group_summary['total_users']
print(group_summary)

  test group  total_users  converted_users  conversion_rate
0         ad       564577            14423         0.025547
1        psa        23524              420         0.017854


In [9]:
# 3. Two-Sample Z-Test for Proportions
counts = group_summary['converted_users'].values
nobs = group_summary['total_users'].values

z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative='two-sided')

# 4. Chi-Square Test of Independence
contingency_table = pd.crosstab(df['test group'], df['converted'])
chi2_stat, chi2_p_val, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nZ-statistic: {z_stat:.4f}, p-value: {p_value:.4e}")
print(f"Chi2 statistic: {chi2_stat:.4f}, p-value: {chi2_p_val:.4e}")


Z-statistic: 7.3701, p-value: 1.7053e-13
Chi2 statistic: 54.0058, p-value: 1.9990e-13


In [10]:
# Bonus. Calculate Absolute Lift & Relative Lift
p_control = group_summary.loc[group_summary['test group'] != 'ad', 'conversion_rate'].values[0]
p_test = group_summary.loc[group_summary['test group'] == 'ad', 'conversion_rate'].values[0]

abs_lift = p_test - p_control
rel_lift = (p_test - p_control) / p_control * 100

print(f"\nAbsolute Lift: {abs_lift * 100:.3f}%")
print(f"Relative Lift: {rel_lift:.2f}%")


Absolute Lift: 0.769%
Relative Lift: 43.09%
